# Pooled pseudobulk computational timing

Aggregate the controlled dataset/method/seed timing records. Each plotted observation is the total wall time for one seed summed across all six production pseudobulk datasets.

💡 **Environment:** `clamp-analyses`

In [ ]:
suppressPackageStartupMessages({
  library(data.table)
  library(ggplot2)
  library(yaml)
  library(here)
  library(scales)
})


In [ ]:
if (exists("snakemake")) {
  timing_paths <- as.character(snakemake@input[["timings"]])
  expected_datasets <- as.character(snakemake@params[["datasets"]])
  expected_methods <- as.character(snakemake@params[["methods"]])
  expected_seeds <- as.integer(snakemake@params[["seeds"]])
  runtime_cfg <- yaml::read_yaml(snakemake@input[["runtime_config"]])$runtime_benchmark
  out_long <- snakemake@output[["long"]]
  out_seed_totals <- snakemake@output[["seed_totals"]]
  out_summary <- snakemake@output[["summary"]]
} else {
  workflow_cfg <- yaml::read_yaml(here("workflow/config/pseudobulk.yaml"))
  runtime_cfg <- yaml::read_yaml(here("workflow/config/runtime_benchmark.yaml"))$runtime_benchmark
  expected_datasets <- names(workflow_cfg$datasets)
  expected_methods <- unlist(runtime_cfg$methods, use.names = FALSE)
  expected_seeds <- as.integer(unlist(runtime_cfg$seeds, use.names = FALSE))
  out_dir <- Sys.getenv("CLAMP_TIMING_ROOT", unset = here(runtime_cfg$output_root))
  timing_paths <- list.files(file.path(out_dir, "timings"), pattern = "[.]csv$",
                            recursive = TRUE, full.names = TRUE)
  out_long <- file.path(out_dir, "runtime_long.csv")
  out_seed_totals <- file.path(out_dir, "runtime_seed_totals.csv")
  out_summary <- file.path(out_dir, "runtime_summary.csv")
}

expected_threads <- as.integer(runtime_cfg$threads)
expected_runs <- length(expected_datasets) * length(expected_methods) * length(expected_seeds)
stopifnot(length(timing_paths) == expected_runs)


## Validate and aggregate all 180 fits

In [ ]:
runtime_long <- rbindlist(lapply(timing_paths, fread), use.names = TRUE, fill = TRUE)
required_columns <- c("dataset", "method", "seed", "threads", "n_genes",
                      "n_samples", "elapsed_seconds", "started_at",
                      "finished_at", "command", "status", "exit_code")
stopifnot(all(required_columns %in% names(runtime_long)))
stopifnot(nrow(runtime_long) == expected_runs)
stopifnot(uniqueN(runtime_long, by = c("dataset", "method", "seed")) == expected_runs)
stopifnot(setequal(runtime_long$dataset, expected_datasets))
stopifnot(setequal(runtime_long$method, expected_methods))
stopifnot(setequal(as.integer(runtime_long$seed), expected_seeds))
stopifnot(all(runtime_long$threads == expected_threads))
stopifnot(all(runtime_long$status == "success"), all(runtime_long$exit_code == 0L))
stopifnot(all(is.finite(runtime_long$elapsed_seconds)), all(runtime_long$elapsed_seconds > 0))

coverage <- runtime_long[, .(n_datasets = uniqueN(dataset), n_runs = .N), by = .(method, seed)]
stopifnot(nrow(coverage) == length(expected_methods) * length(expected_seeds))
stopifnot(all(coverage$n_datasets == length(expected_datasets)))
stopifnot(all(coverage$n_runs == length(expected_datasets)))

runtime_seed_totals <- runtime_long[, .(
  total_seconds = sum(elapsed_seconds),
  total_minutes = sum(elapsed_seconds) / 60,
  n_datasets = uniqueN(dataset),
  n_runs = .N
), by = .(method, seed)]
stopifnot(nrow(runtime_seed_totals) == 30L)
stopifnot(all(runtime_seed_totals$n_datasets == 6L), all(runtime_seed_totals$n_runs == 6L))

runtime_summary <- runtime_seed_totals[, .(
  n_seeds = .N,
  median_minutes = median(total_minutes),
  q1_minutes = as.numeric(quantile(total_minutes, 0.25)),
  q3_minutes = as.numeric(quantile(total_minutes, 0.75)),
  min_minutes = min(total_minutes),
  max_minutes = max(total_minutes)
), by = method]
stopifnot(nrow(runtime_summary) == 10L, all(runtime_summary$n_seeds == 3L))
timing_method_order <- runtime_summary[order(median_minutes), method]
runtime_seed_totals[, method := factor(method, levels = timing_method_order)]
setorder(runtime_seed_totals, method, seed)
setorder(runtime_summary, median_minutes)

for (path in c(out_long, out_seed_totals, out_summary)) {
  dir.create(dirname(path), recursive = TRUE, showWarnings = FALSE)
}
fwrite(runtime_long, out_long)
fwrite(runtime_seed_totals, out_seed_totals)
fwrite(runtime_summary, out_summary)


## Total runtime across the six datasets

In [ ]:
figure_cfg <- yaml::read_yaml(here("config.yaml"))
method_colors <- unlist(figure_cfg$MODEL_COLORS)
names(method_colors)[names(method_colors) == "GenomicSuperSignature"] <- "GSSig"
runtime_seed_totals[, method := factor(method, levels = timing_method_order)]

runtime_plot <- ggplot(runtime_seed_totals, aes(method, total_minutes, fill = method)) +
  geom_boxplot(width = 0.58, outlier.shape = NA, color = "black", linewidth = 0.35) +
  geom_point(aes(shape = factor(seed)), position = position_jitter(width = 0.08, height = 0),
             size = 2.2, fill = "white", color = "black", stroke = 0.5) +
  scale_fill_manual(values = method_colors, na.value = "grey70") +
  scale_shape_manual(values = c(`123` = 21, `456` = 22, `789` = 24), name = "Seed") +
  scale_y_log10(breaks = scales::log_breaks(n = 6), labels = scales::label_number()) +
  labs(x = NULL, y = "Total wall time across six datasets (min)") +
  theme_classic(base_size = 9, base_family = "Helvetica") +
  theme(axis.text.x = element_text(angle = 40, hjust = 1),
        legend.position = "top", legend.title = element_text(face = "plain"))

options(repr.plot.width = 7.2, repr.plot.height = 3.15)
runtime_plot
